In [25]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 9.6 MB/s  0:00:00ta 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 1.6 MB/s  0:00:12 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn]0m 5/6 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [26]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [10]:
#creates Path object pointing to folder
data_path = Path("/Users/kierachiu/Documents/IDX Exchange/IDX-Exchange-Data-Science/data/csv")

#creates empty list to store wanted files
files = []

for file in data_path.glob("CRMLSSold*.csv"): #looking for specific format
    if re.fullmatch(r"CRMLSSold\d{6}(?:_filled)?.csv", file.name):
        files.append(file)

#sort files by date
files = sorted(files)

print(f"Found {len(files)} files")

for file in files:
    print(file.name)

Found 28 files
CRMLSSold202401_filled.csv
CRMLSSold202402.csv
CRMLSSold202403_filled.csv
CRMLSSold202404_filled.csv
CRMLSSold202405_filled.csv
CRMLSSold202406_filled.csv
CRMLSSold202407_filled.csv
CRMLSSold202408.csv
CRMLSSold202409.csv
CRMLSSold202410.csv
CRMLSSold202411.csv
CRMLSSold202412.csv
CRMLSSold202501_filled.csv
CRMLSSold202502.csv
CRMLSSold202503.csv
CRMLSSold202504.csv
CRMLSSold202505.csv
CRMLSSold202506.csv
CRMLSSold202507.csv
CRMLSSold202508.csv
CRMLSSold202509.csv
CRMLSSold202510.csv
CRMLSSold202511.csv
CRMLSSold202512.csv
CRMLSSold202601.csv
CRMLSSold202602.csv
CRMLSSold202603.csv
CRMLSSold202604.csv


In [11]:
# stores all modified CSVs in a list
dfs = []

for file in files:
    df_temp = pd.read_csv(file, low_memory=False)

    # create column to track original CSV file name
    df_temp["source_file"] = file.name

    # extract YYYYMM from filename
    month_match = re.search(r"\d{6}", file.name)

    if month_match:
        # create new column and add data_month 
        df_temp["data_month"] = month_match.group()

    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True) #discards old indexes after concatenating

# Filter for only Residential, SingleFamilyResidence 
df_model = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")].copy()

print("Original rows:", len(df))
print("Single-family residential rows:", len(df_model))

Original rows: 615707
Single-family residential rows: 309735


In [12]:
target = "ClosePrice"

categorical_features = ["City"]

numeric_features = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet"]

features = numeric_features + categorical_features

# make sure all features exist
for col in [target] + features:
    print(col, "->", col in df.columns)

missing_summary = df[[target] + features].isna().sum().to_frame("missing_count")

missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(df) * 100)

missing_summary.sort_values("missing_percent", ascending=False)

ClosePrice -> True
LivingArea -> True
BedroomsTotal -> True
BathroomsTotalInteger -> True
LotSizeSquareFeet -> True
PropertyType -> True
PropertySubType -> True
City -> True


,missing_count,missing_percent
LotSizeSquareFeet,56597,9.192197
PropertySubType,47066,7.644220
LivingArea,43294,7.031591
BedroomsTotal,40834,6.632051
BathroomsTotalInteger,28375,4.608523
City,441,0.071625
ClosePrice,7,0.001137
PropertyType,0,0.000000


In [21]:
# convert ClosePrice into a number, replace with NaN if can't convert value to number
df_model["ClosePrice"] = pd.to_numeric(df_model["ClosePrice"], errors="coerce")

# keep rows that are not missing, and sale price is greater than 0 
df_model = df_model[df_model["ClosePrice"].notna() & (df_model["ClosePrice"] > 0)].copy()

print("Rows remaining:", len(df_model))

# convert all numeric feature columns to a number
for col in numeric_features:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# any invalid LivingArea, LotSizeSquareFeet values converted to NaN
df_model.loc[df_model["LivingArea"] <= 0, "LivingArea"] = np.nan
df_model.loc[df_model["LotSizeSquareFeet"] <= 0, "LotSizeSquareFeet"] = np.nan

categorical_features = ["City"]

for col in categorical_features:
    # convert values to pandas string type
    df_model[col] = df_model[col].astype("string")

    # remove extra spaces at the beginning and end
    df_model[col] = df_model[col].str.strip()

    # make capitalization consistent
    df_model[col] = df_model[col].str.title()

    # replace empty strings with missing values
    df_model[col] = df_model[col].replace("", pd.NA)

# print out the percentage of missing values for each feature
print(df_model[features].isna().mean().sort_values(ascending=False) * 100)
df_model[features + [target, "data_month"]].head()

Rows remaining: 309732
LotSizeSquareFeet        1.772177
LivingArea               0.094275
City                     0.078778
BathroomsTotalInteger    0.016143
BedroomsTotal            0.000000
PropertyType             0.000000
PropertySubType          0.000000
dtype: float64


,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,PropertyType,PropertySubType,City,ClosePrice,data_month
5,1974.0,4.0,4.0,NaN,Residential,SingleFamilyResidence,National City,815000.0,202401
9,1974.0,4.0,4.0,NaN,Residential,SingleFamilyResidence,National City,810000.0,202401
28,3736.0,4.0,4.0,11219.0,Residential,SingleFamilyResidence,San Luis Obispo,2100000.0,202401
29,2100.0,3.0,0.0,74487.6,Residential,SingleFamilyResidence,Fort Bragg,1950000.0,202401
31,2442.0,4.0,3.0,8712.0,Residential,SingleFamilyResidence,San Jose,2340000.0,202401


In [23]:
# get and sort all unique months
months = sorted(df_model["data_month"].unique())

test_month = months[-1] # most recent month

print("Test month:", test_month)

X_months = 6
test_index = months.index(test_month) # get index of test month and months in list

train_months = months[max(0, test_index - X_months):test_index] # get 6 months before the test month

print("Training months:", train_months)
print("Test month:", test_month)

# create train dataset using only the train_months
train_df = df_model[df_model["data_month"].isin(train_months)].copy()
# create test dataset using only the test_month
test_df = df_model[df_model["data_month"] == test_month].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

Test month: 202604
Training months: ['202510', '202511', '202512', '202601', '202602', '202603']
Test month: 202604
Training rows: 60234
Test rows: 12490


In [28]:

X_train = train_df[features] # contains input columns/features to predict only 
y_train = train_df[target] # contains sales price

X_test = test_df[features] # informaiton used to test model
y_test = test_df[target] # sales price for evaluating

""" Imputer - fills missing values with median of column
    Scaler - standardizes each numericl column so values are on similar scale
"""
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), 
            ("scaler", StandardScaler())])

""" Imputer - missing values replaced with unknown
    Encoder - OneHotEncoder converts categories into numeric indicator columns
"""
categorical_transformer = Pipeline(
    steps=[
        ("imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown")),
        ("encoder",
            OneHotEncoder(
                handle_unknown="ignore"))])

"""apply numeric transformer to columns in numberic_features
   apply categorical transformer to columns in categorical features
"""
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric",
             numeric_transformer,
             numeric_features),
        ("categorical",
            categorical_transformer,
            categorical_features)])